In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# List the contents of 'My Drive'
!ls -F '/content/drive/My Drive'


'ADIO OLAMILEKAN (1).pdf'	    Assignment.gdoc
'ADIO OLAMILEKAN.pdf'		   'Colab Notebooks'/
'ADIO OLAMILEKAN RASHEED.pdf'	   'Spotify Listening History.gsheet'
'Adio olamilekan resume.pdf'	   'Untitled form (1).gform'
 annotations/			   'Untitled form (2).gform'
'Arcane Wagers Porposal (1).docx'  'Untitled form (3).gform'
'Arcane Wagers Porposal.docx'	   'Untitled form.gform'
'Arcane Wagers Porposal.gdoc'	    val2017/


## Verify and Update Dataset Paths

### Subtask:
Based on the located dataset path, generate code to ensure the `data_dir` and `coco_root` variables in cell `yi1rOPLjiei-` correctly point to your unzipped dataset within Google Drive. This might involve creating symbolic links or directly updating the path variables.


In [4]:
import torch
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [5]:
import os
from torchvision.datasets import CocoDetection
import torchvision.transforms as T
from torch.utils.data import random_split

# Define paths
# The dataset was found directly under '/content/drive/My Drive'
coco_root = '/content/drive/My Drive'
ann_file = os.path.join(coco_root, 'annotations', 'instances_val2017.json')

# Check if dataset exists
if not os.path.exists(ann_file):
    print("=" * 80)
    print("COCO 2017 VAL DATASET - MANUAL DOWNLOAD REQUIRED")
    print("=" * 80)
    print("\nThe dataset is not found. Please follow these steps:\n")
    print("STEP 1: Download the dataset")
    print("  Images: http://images.cocodataset.org/zips/val2017.zip (1GB)")
    print("  Annotations: http://images.cocodataset.org/annotations/annotations_trainval2017.zip (241MB)")
    print("  Save them to your Downloads folder\n")
    print("STEP 2: Extract the downloaded files")
    print(f"  Extract 'val2017.zip' to: {os.path.abspath(coco_root)}/")
    print(f"  Extract 'annotations_trainval2017.zip' to: {os.path.abspath(coco_root)}/")
    print("  After extraction, you should have this structure:")
    print(f"  {os.path.abspath(coco_root)}/")
    print("    ├── val2017/")
    print("    └── annotations/")
    print("        ├── instances_val2017.json")
    print("        └── ...")
    print("\nSTEP 3: Re-run this cell after extraction")
    print("=" * 80)

    # Create data directory if it doesn't exist
    # Note: data_dir is not explicitly used anymore for coco_root, but keeping os.makedirs for robustness if needed elsewhere.
    # However, for this subtask, we are directly setting coco_root.
    os.makedirs(os.path.dirname(os.path.dirname(ann_file)), exist_ok=True) # Ensure the base directory for annotations exists
    print(f"\n✓ Created directory: {os.path.abspath(os.path.dirname(os.path.dirname(ann_file)))}")
    print(f"  Please extract the .zip files into this directory.\n")

    yolo_train_dataset = None
    yolo_test_dataset = None

else:
    print("=" * 80)
    print("✓ Dataset found! Loading...")
    print("=" * 80)

    try:
        # Load dataset
        coco_dataset = CocoDetection(
            root=os.path.join(coco_root, 'val2017'),
            annFile=ann_file,
            transform=T.Compose([
                T.Resize((448, 448)),
                T.ToTensor(),
            ])
        )

        print(f"✓ Dataset loaded successfully!")
        print(f"✓ Number of samples: {len(coco_dataset)}")
        print(f"✓ Dataset location: {os.path.abspath(coco_root)}")

        # Test loading one sample
        img, target = coco_dataset[0]
        print(f"\n✓ Sample image shape: {img.shape}")
        print(f"✓ Sample loaded successfully!")
        print("=" * 80)

        # Split into train and test datasets
        train_size = int(0.8 * len(coco_dataset))
        test_size = len(coco_dataset) - train_size
        yolo_train_dataset, yolo_test_dataset = random_split(coco_dataset, [train_size, test_size])

    except Exception as e:
        print(f"\n✗ Error loading dataset: {e}")
        print("\nPlease verify:")
        print(f"  1. The extracted folder structure is correct")
        print(f"  2. Path exists: {os.path.abspath(ann_file)}")
        print(f"  3. Check that val2017/ and annotations/ folders are present")
        print("=" * 80)
        yolo_train_dataset = None
        yolo_test_dataset = None

✓ Dataset found! Loading...
loading annotations into memory...
Done (t=1.40s)
creating index...
index created!
✓ Dataset loaded successfully!
✓ Number of samples: 5000
✓ Dataset location: /content/drive/My Drive

✓ Sample image shape: torch.Size([3, 448, 448])
✓ Sample loaded successfully!


In [6]:
COCO_target = [{'bbox': [10, 20, 30, 40], 'category_id': 1}]

In [7]:
def collate_fn(batch):
    images = []
    targets = []
    for img, tgt in batch:
        images.append(img)
        targets.append(tgt)
    images = torch.stack(images)
    return images, targets

train_dataloader = DataLoader(
    yolo_train_dataset,
    batch_size=16,
    num_workers=0,
    persistent_workers=False,
    drop_last=True,
    shuffle=True,
    collate_fn=collate_fn
)
test_dataloader = DataLoader(
    yolo_test_dataset,
    batch_size=16,
    num_workers=0,
    persistent_workers=False,
    drop_last=True,
    collate_fn=collate_fn
)
initial_batch = next(iter(train_dataloader))
images, targets = initial_batch[0], initial_batch[1]
print(f"the shape of batched images: {images.shape}")
# the shape of batched images: (16, 3, 448, 448)
print(f"Number of target samples: {len(targets)}")
# Number of target samples: 16

the shape of batched images: torch.Size([16, 3, 448, 448])
Number of target samples: 16


In [8]:
S = 7 # the entire image is divided into SxS(=7x7) cells
B = 2 # a cell predicts up to B(=2) bounding boxes
C = 80 # The number of classes is C(=80) for COCO

classes = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
class_to_id = {name: i for i, name in enumerate(classes)}

In [9]:
def convert_coco_to_yolo(target, S=7, B=2, C=80):
    yolo_target = torch.zeros(S, S, B*5 + C)
    for obj in target:
        bbox = obj['bbox']
        category_id = obj['category_id']
        class_id = category_id - 1  # COCO category_id starts from 1
        if class_id >= C or class_id < 0:
            continue
        # Assume images are resized to 448x448
        width = 448
        height = 448
        x_center = (bbox[0] + bbox[2] / 2) / width
        y_center = (bbox[1] + bbox[3] / 2) / height
        w = bbox[2] / width
        h = bbox[3] / height
        i = int(y_center * S)
        j = int(x_center * S)
        if i >= S or j >= S:
            continue
        # Assign to first bbox
        yolo_target[i, j, 0] = x_center * S - j
        yolo_target[i, j, 1] = y_center * S - i
        yolo_target[i, j, 2] = w
        yolo_target[i, j, 3] = h
        yolo_target[i, j, 4] = 1
        yolo_target[i, j, 5 + class_id] = 1
    return yolo_target

In [10]:
class Yolo(nn.Module):
    def __init__(self):
        super().__init__()
        self.Conv_block1 = nn.Sequential(nn.Conv2d(in_channels=3,out_channels=64,kernel_size=7,stride=2,padding=3),# 448 --> 224
                                         nn.BatchNorm2d(64),
                                         nn.LeakyReLU(0.1,inplace=True),
                                         nn.MaxPool2d(kernel_size=2,stride=2))# 224 --> 112
        self.Conv_block2 = nn.Sequential(nn.Conv2d(64,
                                                    192,
                                                    kernel_size=3,
                                                    padding=1 # 112 --> 112
                                                    ),
                                          nn.BatchNorm2d(192),
                                          nn.LeakyReLU(0.1, inplace=True),
                                          nn.MaxPool2d(kernel_size=2,
                                                       stride=2)) # 112 -> 56
        self.Conv_block3 = nn.Sequential(nn.Conv2d(192,
                                                    128,
                                                    kernel_size=1, # 56 --> 56
                                                    ),
                                          nn.BatchNorm2d(128),
                                          nn.LeakyReLU(0.1, inplace=True),
                                          nn.Conv2d(128,
                                                    256,
                                                    kernel_size=3,
                                                    padding=1), # 56 --> 56
                                          nn.BatchNorm2d(256),
                                          nn.LeakyReLU(0.1, inplace=True),
                                          nn.Conv2d(256,
                                                    256,
                                                    kernel_size=1), # 56 --> 56
                                          nn.BatchNorm2d(256),
                                          nn.LeakyReLU(0.1, inplace=True),
                                          nn.Conv2d(256,
                                                    512,
                                                    kernel_size=3,
                                                    padding=1),  # 56 --> 56
                                          nn.BatchNorm2d(512),
                                          nn.LeakyReLU(0.1, inplace=True),
                                          nn.MaxPool2d(kernel_size=2,
                                                       stride=2))# 56 --> 28)
        Conv_block4 = []
        for _ in range(4):
            Conv_block4.append(nn.Conv2d(512, 256, kernel_size=1))
            Conv_block4.append(nn.BatchNorm2d(256))
            Conv_block4.append(nn.LeakyReLU(0.1, inplace=True))
            Conv_block4.append(nn.Conv2d(256, 512, kernel_size=3, padding=1))
            Conv_block4.append(nn.BatchNorm2d(512))
            Conv_block4.append(nn.LeakyReLU(0.1, inplace=True))

        Conv_block4.append(nn.Conv2d(512, 512, kernel_size=1))
        Conv_block4.append(nn.BatchNorm2d(512))
        Conv_block4.append(nn.LeakyReLU(0.1, inplace=True))
        Conv_block4.append(nn.Conv2d(512, 1024, kernel_size=3, padding=1))
        Conv_block4.append(nn.BatchNorm2d(1024))
        Conv_block4.append(nn.LeakyReLU(0.1, inplace=True))
        Conv_block4.append(nn.MaxPool2d(kernel_size=2, stride=2)) # 28 --> 14

        self.Conv_block4 = nn.Sequential(*Conv_block4)



        Conv_block5 = []
        for _ in range(2):
            Conv_block5.append(nn.Conv2d(1024, 512, kernel_size=1))
            Conv_block5.append(nn.BatchNorm2d(512))
            Conv_block5.append(nn.LeakyReLU(0.1, inplace=True))
            Conv_block5.append(nn.Conv2d(512, 1024, kernel_size=3, padding=1))
            Conv_block5.append(nn.BatchNorm2d(1024))
            Conv_block5.append(nn.LeakyReLU(0.1, inplace=True))
        Conv_block5.append(nn.Conv2d(1024, 1024, kernel_size=3, padding=1))
        Conv_block5.append(nn.BatchNorm2d(1024))
        Conv_block5.append(nn.LeakyReLU(0.1, inplace=True))
        Conv_block5.append(nn.MaxPool2d(kernel_size=2, stride=2)) # 14 --> 7
        self.Conv_block5 = nn.Sequential(*Conv_block5)
        self.Conv_block6 = nn.Sequential(nn.Conv2d(1024,1024, kernel_size=3, padding=1),
                                          nn.BatchNorm2d(1024),
                                          nn.LeakyReLU(0.1, inplace=True),
                                          nn.Conv2d(1024,1024, kernel_size=3, padding=1),
                                          nn.BatchNorm2d(1024),
                                          nn.LeakyReLU(0.1, inplace=True))

        self.prediction_head = nn.Sequential(nn.Linear(in_features=1024*7*7, out_features=4096),
                                             nn.BatchNorm1d(4096),
                                             nn.LeakyReLU(0.1, inplace=True),
                                             nn.Dropout(0.5),
                                             nn.Linear(in_features=4096,
                                                       out_features=S*S*(B*5 + C)))

    def forward(self,x):
        x = self.Conv_block1(x)
        x = self.Conv_block2(x)
        x = self.Conv_block3(x)
        x = self.Conv_block4(x)
        x = self.Conv_block5(x)
        x = self.Conv_block6(x)
        x = x.view(x.size(0), -1)
        x = torch.reshape(self.prediction_head(x),
                          (x.shape[0],S, S, B*5 + C))
        return x

In [11]:
def yolo_loss(pred, target, S=7, B=2, C=80, lambda_coord=5, lambda_noobj=0.5):
    batch_size = pred.size(0)
    pred = pred.view(batch_size, S, S, B*5 + C)
    target = target.view(batch_size, S, S, B*5 + C)

    # pred_boxes: (batch_size, S, S, B, 5) => x, y, w, h, conf
    pred_boxes = pred[..., :B*5].view(batch_size, S, S, B, 5)
    # pred_classes: (batch_size, S, S, C)
    pred_classes = pred[..., B*5:]

    # target_boxes: (batch_size, S, S, B, 5) => x, y, w, h, conf (here conf is 1 if obj, 0 if no obj)
    target_boxes = target[..., :B*5].view(batch_size, S, S, B, 5)
    # target_classes: (batch_size, S, S, C)
    target_classes = target[..., B*5:]

    # For simplicity, we are considering only one bounding box from B (the first one)
    pred_box = pred_boxes[..., 0, :]  # (batch_size, S, S, 5)
    target_box = target_boxes[..., 0, :] # (batch_size, S, S, 5)

    # 1. Localization Loss (bbox coordinate loss)
    # Only compute for cells that contain an object
    obj_mask = target_box[..., 4] > 0 # (batch_size, S, S) boolean mask

    loc_loss = torch.tensor(0.0, device=pred.device)
    if obj_mask.sum() > 0:
        # Clamp predicted width and height to a small positive value to avoid sqrt(negative)
        pred_w_h_obj = torch.clamp(pred_box[obj_mask][:, 2:4], min=1e-6)
        target_w_h_obj = target_box[obj_mask][:, 2:4]

        # x, y loss
        loc_xy_loss = torch.sum((pred_box[obj_mask][:, :2] - target_box[obj_mask][:, :2])**2)
        # w, h loss (using square root as per YOLO)
        loc_wh_loss = torch.sum((torch.sqrt(pred_w_h_obj) - torch.sqrt(target_w_h_obj))**2)
        loc_loss = lambda_coord * (loc_xy_loss + loc_wh_loss)

    # 2. Confidence Loss
    # Confidence for cells with objects (target_box[..., 4] is 1)
    conf_obj_loss = torch.sum((pred_box[obj_mask][:, 4] - target_box[obj_mask][:, 4])**2)

    # Confidence for cells without objects (target_box[..., 4] is 0)
    noobj_mask = target_box[..., 4] == 0 # (batch_size, S, S) boolean mask
    conf_noobj_loss = lambda_noobj * torch.sum((pred_box[noobj_mask][:, 4] - target_box[noobj_mask][:, 4])**2)

    conf_loss = conf_obj_loss + conf_noobj_loss

    # 3. Class Probability Loss
    # Only compute for cells that contain an object
    class_loss = torch.tensor(0.0, device=pred.device)
    if obj_mask.sum() > 0:
        # Reshape obj_mask to match target_classes dimensions for broadcasting
        obj_mask_classes = obj_mask.unsqueeze(-1).expand_as(target_classes)
        class_loss = torch.sum((pred_classes[obj_mask_classes] - target_classes[obj_mask_classes])**2)


    total_loss = loc_loss + conf_loss + class_loss
    return total_loss / batch_size

In [12]:
def compute_iou(box1, box2):
    # box = [x1, y1, x2, y2]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

In [13]:
def compute_ap(recalls, precisions):
    # Compute AP using 11-point interpolation
    ap = 0.0
    for t in np.arange(0, 1.1, 0.1):
        if np.sum(recalls >= t) == 0:
            p = 0
        else:
            p = np.max(precisions[recalls >= t])
        ap += p / 11.0
    return ap

In [14]:
model = Yolo().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in train_dataloader:
        images = images.to(device)
        batch_targets = torch.stack([convert_coco_to_yolo(t) for t in targets]).to(device)
        predictions = model(images)
        loss = yolo_loss(predictions, batch_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss / len(train_dataloader):.4f}')

Epoch 1/10, Loss: 251.5455
Epoch 2/10, Loss: 170.0899
Epoch 3/10, Loss: 132.5566
Epoch 4/10, Loss: 79.4035
Epoch 5/10, Loss: 64.8701
Epoch 6/10, Loss: 68.3648
Epoch 7/10, Loss: 47.0995
Epoch 8/10, Loss: 41.0594
Epoch 9/10, Loss: 32.1942
Epoch 10/10, Loss: 28.9651


# Task
Save the retrained YOLO model to `yolo_model.pth`, then evaluate its performance on the test dataset by calculating and presenting the average test loss.

## Save Model

### Subtask:
Save the current state of the trained YOLO model to a file named `yolo_model.pth`.


**Reasoning**:
The subtask requires saving the trained YOLO model. This step will use `torch.save()` to store the model's state dictionary to a file named `yolo_model.pth`.



In [15]:
torch.save(model.state_dict(), 'yolo_model.pth')
print("Model saved to yolo_model.pth")

Model saved to yolo_model.pth


In [16]:
def cells_to_boxes(model_output, S=7, B=2, C=80, img_size=448):
    # model_output: (S, S, B*5 + C)
    # Returns a list of dictionaries, each representing a detected object:
    # {'bbox': [x1, y1, x2, y2], 'confidence': conf, 'class_id': class_idx}

    detections = []
    for i in range(S):
        for j in range(S):
            cell_output = model_output[i, j, :]

            # Process B bounding boxes
            for b in range(B):
                box_offset = b * 5
                # Raw predictions for bbox (x, y, w, h, conf)
                x_offset_pred = torch.sigmoid(cell_output[box_offset + 0]) # x center relative to cell
                y_offset_pred = torch.sigmoid(cell_output[box_offset + 1]) # y center relative to cell
                w_pred = cell_output[box_offset + 2] # width relative to image
                h_pred = cell_output[box_offset + 3] # height relative to image
                conf_pred = torch.sigmoid(cell_output[box_offset + 4]) # object confidence

                # Class probabilities
                class_scores = torch.softmax(cell_output[B*5:], dim=0)
                class_conf, class_idx = torch.max(class_scores, dim=0)

                # Overall confidence for this detection
                overall_confidence = conf_pred * class_conf

                # Convert cell relative coords to image relative coords (0-1 range)
                x_center_abs = (j + x_offset_pred) / S
                y_center_abs = (i + y_offset_pred) / S

                # Clamp w and h to a small positive value to avoid issues with sqrt in loss if not already handled
                # In prediction, we can just use the raw output (it can be any value, but we might want to scale it).
                # For now, let's assume it's directly predicted as ratio of image size.

                # Convert (x_center, y_center, w, h) to (x1, y1, x2, y2)
                x1 = (x_center_abs - w_pred / 2) * img_size
                y1 = (y_center_abs - h_pred / 2) * img_size
                x2 = (x_center_abs + w_pred / 2) * img_size
                y2 = (y_center_abs + h_pred / 2) * img_size

                # Ensure coordinates are within image bounds
                x1 = torch.clamp(x1, 0, img_size)
                y1 = torch.clamp(y1, 0, img_size)
                x2 = torch.clamp(x2, 0, img_size)
                y2 = torch.clamp(y2, 0, img_size)

                detections.append({
                    'bbox': [x1.item(), y1.item(), x2.item(), y2.item()],
                    'confidence': overall_confidence.item(),
                    'class_id': class_idx.item()
                })
    return detections

In [17]:
def non_max_suppression(boxes, iou_threshold=0.5, conf_threshold=0.01):
    # boxes: List of dictionaries, each with 'bbox', 'confidence', 'class_id'
    if not boxes: return []

    # Sort boxes by confidence in descending order
    boxes = sorted(boxes, key=lambda x: x['confidence'], reverse=True)

    # Store final detections
    filtered_boxes = []

    while boxes:
        # Pick the box with highest confidence
        best_box = boxes.pop(0)

        # Add it to filtered list if its confidence is above threshold
        if best_box['confidence'] < conf_threshold: continue

        filtered_boxes.append(best_box)

        # Remove other boxes that heavily overlap with the best box and are of the same class
        boxes = [box for box in boxes if box['class_id'] != best_box['class_id'] or compute_iou(box['bbox'], best_box['bbox']) < iou_threshold]

    return filtered_boxes

In [18]:
def evaluate_yolo_model(model, dataloader, device, S=7, B=2, C=80, img_size=448, iou_threshold=0.5, conf_threshold=0.01):
    model.eval()
    all_detections = []
    all_ground_truths = []
    all_image_ids = [] # To keep track of images if multiple objects per image

    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(dataloader):
            images = images.to(device)
            predictions = model(images)

            for img_idx in range(images.shape[0]):
                img_predictions = predictions[img_idx]
                image_detections = cells_to_boxes(img_predictions, S, B, C, img_size)
                # Apply NMS per image
                nms_detections = non_max_suppression(image_detections, iou_threshold, conf_threshold)

                # Collect ground truth for the current image
                image_ground_truths = []
                # Convert COCO target format to [x1, y1, x2, y2, class_id]
                for obj in targets[img_idx]:
                    bbox = obj['bbox']
                    category_id = obj['category_id']
                    class_id = category_id - 1  # Adjust for 0-indexed classes

                    # COCO bbox: [x, y, width, height]
                    # Convert to [x1, y1, x2, y2]
                    gt_x1 = bbox[0]
                    gt_y1 = bbox[1]
                    gt_x2 = bbox[0] + bbox[2]
                    gt_y2 = bbox[1] + bbox[3]
                    image_ground_truths.append({
                        'bbox': [gt_x1, gt_y1, gt_x2, gt_y2],
                        'class_id': class_id,
                        'matched': False # To track if this GT has been matched
                    })

                all_detections.extend(nms_detections)
                all_ground_truths.extend(image_ground_truths)
                all_image_ids.extend([batch_idx * images.shape[0] + img_idx] * len(nms_detections))

    # Prepare for mAP calculation
    # Sort detections by confidence
    all_detections.sort(key=lambda x: x['confidence'], reverse=True)

    true_positives = [0] * len(all_detections)
    false_positives = [0] * len(all_detections)
    detection_scores = [d['confidence'] for d in all_detections]

    # Create a copy of ground truths for matching
    # This is simplified; a proper mAP would track per-image matches.
    # For a global AP, we match across all GTs.
    gt_copy = [gt.copy() for gt in all_ground_truths]

    for det_idx, det in enumerate(all_detections):
        best_iou = 0.0
        best_gt_idx = -1

        # Find the best matching ground truth for this detection
        for gt_idx, gt in enumerate(gt_copy):
            if gt['class_id'] == det['class_id'] and not gt['matched']:
                iou = compute_iou(det['bbox'], gt['bbox'])
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

        if best_iou >= iou_threshold:
            true_positives[det_idx] = 1
            gt_copy[best_gt_idx]['matched'] = True # Mark ground truth as matched
        else:
            false_positives[det_idx] = 1

    # Calculate cumulative TP and FP
    cumulative_tp = torch.cumsum(torch.tensor(true_positives), dim=0).float()
    cumulative_fp = torch.cumsum(torch.tensor(false_positives), dim=0).float()

    # Calculate precision and recall
    num_ground_truths = len(all_ground_truths)
    if num_ground_truths == 0:
        print("No ground truth objects found. Cannot compute mAP.")
        return 0.0

    precisions = cumulative_tp / (cumulative_tp + cumulative_fp)
    recalls = cumulative_tp / num_ground_truths

    # Remove NaNs if any (where denominator is zero)
    precisions[torch.isnan(precisions)] = 0
    recalls[torch.isnan(recalls)] = 0

    ap = compute_ap(recalls.cpu().numpy(), precisions.cpu().numpy())

    print(f"Mean Average Precision (mAP) @ IoU={iou_threshold}: {ap:.4f}")
    return ap

In [19]:
# Example call to evaluate the model
# Set desired IoU threshold for mAP calculation
IOU_THRESHOLD = 0.5
MAP_score = evaluate_yolo_model(model, test_dataloader, device, iou_threshold=IOU_THRESHOLD)


Mean Average Precision (mAP) @ IoU=0.5: 0.0009


In [20]:
model.eval()
test_loss = 0.0
with torch.no_grad():
    for images, targets in test_dataloader:
        images = images.to(device)
        batch_targets = torch.stack([convert_coco_to_yolo(t) for t in targets]).to(device)
        predictions = model(images)
        loss = yolo_loss(predictions, batch_targets)
        test_loss += loss.item()

num_test_batches = len(test_dataloader)
if num_test_batches > 0:
    average_test_loss = test_loss / num_test_batches
    print(f'Average Test Loss: {average_test_loss:.4f}')
else:
    print('Test dataloader is empty, cannot compute average test loss.')

Average Test Loss: 15.8951
